## Colab XGBoost-GPU training (reads from local HDFS via ngrok + HttpFS)

This notebook trains an XGBoost model (GPU) using baseline feature Parquet stored in your local HDFS, and writes model artifacts back to HDFS.

### Architecture

- **Laptop (local stack)**
  - HDFS is running locally (namenode + multiple datanodes)
  - **HttpFS** provides a single HTTP gateway that implements the WebHDFS REST API **without datanode redirects**
  - ngrok exposes **one** public URL to HttpFS (port **14000**)

- **Colab (GPU runtime)**
  - Uses WebHDFS REST only (via HttpFS; no JNI / libhdfs)
  - Reads `/supply-chain/features/baseline/{commodity}/` from Spark output
  - Creates train/val/test splits inside Colab
  - Writes artifacts back to HDFS under `/supply-chain/models/`, `/supply-chain/predictions/`, `/supply-chain/model_metrics/`

### Order of operations

1. On your laptop:

```bash
./scripts/start_hdfs_tunnel.sh
```

2. Copy the printed `https://...ngrok-free.app|dev` URL and paste it into `HDFS_WEBHDFS_URL` below.

3. In Colab: set **Runtime → Change runtime type → GPU**.

4. Run all notebook cells.


In [ ]:
# Params
# Set this from `./scripts/start_hdfs_tunnel.sh` output (ngrok prints an https://... URL)
# This should point to HttpFS (port 14000). No trailing slash.
HDFS_WEBHDFS_URL = "https://YOUR-HTTPFS.ngrok-free.app"
HDFS_USER = "root"

COMMODITY = "brent"

# Read the Spark-produced baseline dataset. Colab creates train/val/test splits.
FEATURES_DIR_HDFS = f"/supply-chain/features/baseline/{COMMODITY}"  # directory in HDFS

# HDFS output locations for artifacts
MODELS_DIR_HDFS = f"/supply-chain/models/{COMMODITY}"
PREDICTIONS_DIR_HDFS = "/supply-chain/predictions"
METRICS_DIR_HDFS = "/supply-chain/model_metrics"


In [ ]:
# Install deps + define a simple WebHDFS wrapper (HttpFS)
!pip install -q hdfs xgboost pandas pyarrow scikit-learn

from __future__ import annotations

import io
import json
from datetime import datetime

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import xgboost as xgb

print(f"XGBoost version: {xgb.__version__}")
try:
    import numpy as np

    d = xgb.DMatrix(np.random.rand(10, 3), label=np.random.randint(0, 3, size=10))
    booster = xgb.train({"tree_method": "hist", "device": "cuda"}, d, num_boost_round=1)
    print("GPU check: OK (trained 1 round on CUDA)")
except Exception as e:
    print("GPU check: FAILED (check Colab runtime is GPU)")
    print(e)


class HdfsBridge:
    """WebHDFS bridge (via HttpFS) for Colab ↔ HDFS.

    HttpFS implements the WebHDFS REST API but does NOT redirect clients to
    datanodes for OPEN/CREATE. That makes it a single-endpoint gateway that
    works cleanly over ngrok.
    """

    def __init__(self, webhdfs_url: str, user: str = "root"):
        from hdfs import InsecureClient as _WebHdfsClient

        self._client = _WebHdfsClient(webhdfs_url, user=user)
        self.url = webhdfs_url
        self.user = user

    def list(self, hdfs_dir: str) -> list[str]:
        return self._client.list(hdfs_dir)

    def mkdirs(self, hdfs_dir: str) -> None:
        self._client.makedirs(hdfs_dir)

    def read_bytes(self, hdfs_path: str) -> bytes:
        with self._client.read(hdfs_path) as reader:
            return reader.read()

    def write_bytes(self, hdfs_path: str, content: bytes, overwrite: bool = True) -> None:
        with self._client.write(hdfs_path, overwrite=overwrite) as writer:
            writer.write(content)

    def write_text(self, hdfs_path: str, text: str, overwrite: bool = True) -> None:
        self.write_bytes(hdfs_path, text.encode(), overwrite=overwrite)


hdfs = HdfsBridge(HDFS_WEBHDFS_URL, user=HDFS_USER)
print("HttpFS WebHDFS:", hdfs.url, "user=", hdfs.user)


In [ ]:
# No Drive mounting in big-data path (HDFS read/write only).


In [ ]:
# List parquet part files in HDFS (no local download)
print("Listing HDFS features dir:", FEATURES_DIR_HDFS)
files = hdfs.list(FEATURES_DIR_HDFS)
parquet_files = [f for f in files if f.endswith(".parquet") or f.startswith("part-")]
print(f"Found {len(parquet_files)} parquet parts")
print(parquet_files[:5])


In [ ]:
# Read baseline parquet parts from HDFS into memory (no local saves)
# NOTE: This materializes the dataset in Colab RAM. For very large datasets,
# you would stream + batch train, or train inside the cluster.

tables = []
for f in parquet_files:
    b = hdfs.read_bytes(f"{FEATURES_DIR_HDFS}/{f}")
    tables.append(pq.read_table(io.BytesIO(b)))

table = pa.concat_tables(tables, promote=True)
df = table.to_pandas()

# Ensure event_date is comparable, then create time-based splits in Colab.
if not pd.api.types.is_datetime64_any_dtype(df["event_date"]):
    df["event_date"] = pd.to_datetime(df["event_date"])

df = df.sort_values("event_date").reset_index(drop=True)
q70 = df["event_date"].quantile(0.70)
q85 = df["event_date"].quantile(0.85)
df["split"] = "test"
df.loc[df["event_date"] < q85, "split"] = "val"
df.loc[df["event_date"] < q70, "split"] = "train"

print(f"Loaded {len(df):,} rows")
print("Splits:")
print(df["split"].value_counts())
df.head()


In [ ]:
# (moved) Parquet read happens in the prior cell


In [ ]:
# Split-aware train/val/test

feature_cols = [c for c in df.columns if c not in ("event_date", "label", "split")]

train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "val"].copy()
test_df = df[df["split"] == "test"].copy()

X_train, y_train = train_df[feature_cols], train_df["label"].astype(int)
X_val, y_val = val_df[feature_cols], val_df["label"].astype(int)
X_test, y_test = test_df[feature_cols], test_df["label"].astype(int)

print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")
print(f"Features: {len(feature_cols):,}")
print("Label counts (train):")
print(y_train.value_counts().sort_index())


In [ ]:
# Train XGBoost (GPU)

model = xgb.XGBClassifier(
    tree_method="hist",
    device="cuda",
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    early_stopping_rounds=50,
    random_state=42,
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=50,
)


In [ ]:
# Evaluate

from sklearn.metrics import classification_report, confusion_matrix, f1_score

labels = [0, 1]
target_names = ["normal", "high_magnitude"]
y_pred_test = model.predict(X_test)

print("Test classification report:")
print(
    classification_report(
        y_test,
        y_pred_test,
        labels=labels,
        target_names=target_names,
        zero_division=0,
    )
)
print(f"\nTest F1 (macro): {f1_score(y_test, y_pred_test, labels=labels, average='macro'):.4f}")

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred_test, labels=labels))


In [ ]:
# Feature importance

imp = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)

imp.head(20)


In [ ]:
# Write artifacts back to HDFS via WebHDFS (no Drive, no local saves)

from datetime import timezone
import numpy as np

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")

# Ensure base dirs exist
hdfs.mkdirs(MODELS_DIR_HDFS)
hdfs.mkdirs(PREDICTIONS_DIR_HDFS)
hdfs.mkdirs(METRICS_DIR_HDFS)

# Model bytes (JSON)
model_bytes = model.get_booster().save_raw(raw_format="json")
hdfs.mkdirs(f"{MODELS_DIR_HDFS}/{timestamp}")
hdfs.write_bytes(f"{MODELS_DIR_HDFS}/{timestamp}/model.json", model_bytes, overwrite=True)

# Predictions parquet bytes
pred_raw = model.predict(X_test)
if isinstance(pred_raw, np.ndarray) and pred_raw.ndim == 2:
    pred_labels = pred_raw.argmax(axis=1)
else:
    pred_labels = pred_raw

preds_df = pd.DataFrame(
    {
        "event_date": test_df["event_date"].to_numpy(),
        "chokepoint": test_df["chokepoint"].to_numpy(),
        "true_label": y_test.to_numpy(),
        "predicted_label": np.asarray(pred_labels).reshape(-1),
    }
)
_buf = io.BytesIO()
pq.write_table(pa.Table.from_pandas(preds_df), _buf)
hdfs.write_bytes(
    f"{PREDICTIONS_DIR_HDFS}/{timestamp}.parquet",
    _buf.getvalue(),
    overwrite=True,
)

# Metrics JSON
labels = [0, 1, 2]
metrics = {
    "timestamp": timestamp,
    "commodity": COMMODITY,
    "train_size": int(len(X_train)),
    "val_size": int(len(X_val)),
    "test_size": int(len(X_test)),
    "test_f1_macro": float(f1_score(y_test, np.asarray(pred_labels).reshape(-1), labels=labels, average="macro")),
    "feature_importance": dict(zip(feature_cols, model.feature_importances_.tolist())),
    "hdfs_features_dir": FEATURES_DIR_HDFS,
}
hdfs.write_text(
    f"{METRICS_DIR_HDFS}/{timestamp}.json",
    json.dumps(metrics, indent=2),
    overwrite=True,
)

# Update CURRENT pointer
hdfs.write_text(f"{MODELS_DIR_HDFS}/CURRENT", timestamp, overwrite=True)

print("Wrote artifacts to HDFS:")
print("  model:", f"{MODELS_DIR_HDFS}/{timestamp}/model.json")
print("  predictions:", f"{PREDICTIONS_DIR_HDFS}/{timestamp}.parquet")
print("  metrics:", f"{METRICS_DIR_HDFS}/{timestamp}.json")
print("  CURRENT:", f"{MODELS_DIR_HDFS}/CURRENT")


### Next steps / troubleshooting (HttpFS)

HttpFS implements the WebHDFS REST API but does **not** redirect clients to datanodes for OPEN/CREATE, which is why it works over a single ngrok endpoint.

Local checks:

```bash
# HttpFS should respond with JSON
curl -sS "http://localhost:14000/webhdfs/v1/?op=LISTSTATUS&user.name=root" | head

# baseline feature listing
curl -sS "http://localhost:14000/webhdfs/v1/supply-chain/features/baseline?op=LISTSTATUS&user.name=root" | head
```

If those work locally but Colab fails:
- confirm ngrok is tunneling **14000**
- confirm `HDFS_WEBHDFS_URL` is exactly the ngrok `https://...` base (no `/webhdfs/...` suffix)
